<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Data Science and Machine Learning</br>
    2.lab.2. Auto MPG Regression
  </strong>
</div>

### Background

You will continue working with the Auto MPG dataset, implementing regression models using scikit-learn.

The dataset includes:
- 398 instances
- 8 attributes including MPG, cylinders, displacement, horsepower, weight, acceleration, model year and origin

### Description

Your task is to implement regression models using scikit-learn to:

- Implement linear regression to predict MPG
- Implement polynomial regression with different degrees
- Compare the performance of linear vs polynomial models
- Visualize the fits for the most relevant feature

### Deliverables

- A *ipynb* (Jupyter Notebook) file called *auto-mpg-regression.ipynb*
- Visualizations comparing linear and polynomial fits
- Analysis of model performance for different polynomial degrees
- Use the provided template as a starting point


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from rich.table import Table
from rich.console import Console

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

Visual configuration of Seaborn for more attractive graphics

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")

Rich console instance to display results

In [ ]:
console = Console()

## Reading Dataset

In [ ]:
from seaborn import load_dataset

df = load_dataset("mpg")

X = df.drop(columns=["mpg"])
y = df["mpg"]

X.head()

## Data cleaning

Convert the origin category to numerical values (0, 1, 2, ...)

In [ ]:
X["origin_numeric"] = pd.Categorical(X["origin"]).codes

Replace null values with the median of the column.

In [ ]:
X["horsepower"] = X["horsepower"].fillna(X["horsepower"].median())

Delete the `name` column because is not informative due to its high number of unique values.

In [ ]:
X = X.drop(columns=["name", "origin"])

### Distribution of label variable

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(y, kde=True, bins=30)
plt.title("Distribution of MPG")
plt.xlabel("Miles per Gallon (MPG)")
plt.ylabel("Frequency")

plt.subplot(1, 2, 2)
sns.boxplot(y)
plt.title("Boxplot of MPG")
plt.ylabel("Miles Per Gallon (MPG)")

plt.tight_layout()
plt.show()

## Preprocessing of data

Split the data into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Standardize features

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

console.print(f"Training set: {X_train.shape}")
console.print(f"Test set: {X_test.shape}")

Method to train a model

In [ ]:
def train_model(X_train, y_train, X_test, y_test, degree=1):

    poly = PolynomialFeatures(degree=degree)
    X_train_poly = poly.fit_transform(X_train)
    X_test_poly = poly.transform(X_test)

    model = LinearRegression()
    model.fit(X_train_poly, y_train)

    y_pred = model.predict(X_test_poly)
    mse = mean_squared_error(y_test, y_pred)

    return model, y_pred, mse

## 1. Lineal Regression

Train and evaluate the model with all features

In [ ]:
_, _, mse_linear = train_model(X_train, y_train, X_test, y_test, degree=1)

console.print(f"MSE Regresión Lineal: {mse_linear:.4f}")

## 2. Polynomial Regression

In [ ]:
table = Table(title="Comparison of polynomial degrees")
table.add_column("Degree", style="green")
table.add_column("MSE", style="cyan")

degrees = [1, 2, 3, 4, 5]

for degree in degrees:
    _, _, mse_polynomial = train_model(X_train, y_train, X_test, y_test, degree)

    table.add_row(str(degree), f"{mse_polynomial:.4f}")

console.print(table)

The best degree option is `2`.

## 3. Compare the performance of linear vs. polynomial models

In [ ]:
table = Table(title="Lineal vs Polynomial")
table.add_column("Type regression", style="green")
table.add_column("MSE", style="cyan")

degrees = [1, 2]

for degree in degrees:
    _, _, mse = train_model(X_train, y_train, X_test, y_test, degree)

    table.add_row("Lineal" if degree == 1 else "Polynomial", f"{mse:.4f}")

console.print(table)

The best option is the `polynomial regression` with a MSE of 6.44. 

## 4. View settings for the most relevant feature

We create a function to render the graphics.

In [ ]:
def render_polynomial_regression(X_test_feature, y_test, y_pred, degree, mse, i):
    plt.subplot(2, 3, i + 1)
    plt.scatter(X_test_feature, y_test, s=30, label="Data")

    sort_idx = np.argsort(X_test_feature[:, 0])
    X_sorted = X_test_feature[sort_idx]
    y_pred_sorted = y_pred[sort_idx]

    plt.plot(X_sorted, y_pred_sorted, color="r", label="Prediction")
    plt.title(
        f"{'Lineal' if degree == 1 else f'Polynomial - Degree {degree}'}, MSE = {mse:.2f}"
    )
    plt.xlabel("Weight")
    plt.ylabel("MPG")
    plt.legend()

We trained the model with polynomial regression. and with each feature, we also verify its MSE value to confirm which is the best feature.

In [ ]:
features_mse = []
degrees = [1, 2, 3]

for i, feature_name in enumerate(X.columns):
    for degree in degrees:
        model, y_pred_feature, mse_feature = train_model(
            X_train[:, [i]], y_train, X_test[:, [i]], y_test, degree
        )

        features_mse.append(
            {"degree": degree, "feature": feature_name, "mse": mse_feature}
        )

features_mse.sort(key=lambda x: x["mse"])

In [ ]:
table = Table(title="Features MSE")

table.add_column("Degree", style="green")
table.add_column("Feature", style="green")
table.add_column("MSE", style="cyan")

for item in features_mse:
    table.add_row(str(item["degree"]), item["feature"], f"{item['mse']:.4f}")


console.print(table)

The best feature is `weight` with the degree `2`.

We compare both models, lineal regression vs polymial regression with the best feature (`weight`)

In [ ]:
degrees = [1, 2]
best_feature = X.columns.get_loc("weight")

plt.figure(figsize=(12, 8))
for i, degree in enumerate(degrees):

    _, y_pred, mse = train_model(
        X_train[:, [best_feature]], y_train, X_test[:, [best_feature]], y_test, degree
    )

    render_polynomial_regression(
        X_test[:, [best_feature]], y_test, y_pred, degree, mse, i
    )

plt.tight_layout()
plt.show()

## Conclusions

- The polynomial model (degree 2) best captures the relationship between the variables and consumption (MPG), whether using a single variable or all of them.
- The best MSE is achieved by using all the features together and not just the best feature.